In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from xgboost import XGBClassifier  #

import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 50)


In [2]:
# Fayl path-ları öz layihənə uyğun dəyiş
price_path = r"C:\Users\orxan\OneDrive\Masaüstü\prediction_hub\PredictBack\predicthub_backend\ml\data\synthetic\price_history.csv"
trades_path = r"C:\Users\orxan\OneDrive\Masaüstü\prediction_hub\PredictBack\predicthub_backend\ml\data\synthetic\trades.csv"
liq_path = r"C:\Users\orxan\OneDrive\Masaüstü\prediction_hub\PredictBack\predicthub_backend\ml\data\synthetic\liquidity_events.csv"

df_price = pd.read_csv(price_path, parse_dates=["timestamp"])
df_trades = pd.read_csv(trades_path, parse_dates=["created_at"])
df_liq = pd.read_csv(liq_path, parse_dates=["created_at"])

df_price.head()


,id,market_id,yes_price,no_price,timestamp
0,372,30,0.483820,0.516180,2025-12-01 13:04:53.037572+00:00
1,371,30,0.518666,0.481334,2025-12-01 08:04:53.037572+00:00
2,107,18,0.428595,0.571405,2025-12-01 01:04:51.908383+00:00
3,370,30,0.532890,0.467110,2025-11-30 23:04:53.037572+00:00
4,369,30,0.499639,0.500361,2025-11-30 16:04:53.037572+00:00


In [3]:
# First, reload and check original columns
price_path = r"C:\Users\orxan\OneDrive\Masaüstü\prediction_hub\PredictBack\predicthub_backend\ml\data\synthetic\price_history.csv"
df_price = pd.read_csv(price_path, parse_dates=["timestamp"])

print("Original columns:", df_price.columns.tolist())
print(df_price.head())

Original columns: ['id', 'market_id', 'yes_price', 'no_price', 'timestamp']
    id  market_id  yes_price  no_price                        timestamp
0  372         30   0.483820  0.516180 2025-12-01 13:04:53.037572+00:00
1  371         30   0.518666  0.481334 2025-12-01 08:04:53.037572+00:00
2  107         18   0.428595  0.571405 2025-12-01 01:04:51.908383+00:00
3  370         30   0.532890  0.467110 2025-11-30 23:04:53.037572+00:00
4  369         30   0.499639  0.500361 2025-11-30 16:04:53.037572+00:00


In [4]:
# Generate realistic synthetic price variation (since original data has static prices)
np.random.seed(42)

# Group by market and add random walk variation to each market's prices
def add_price_variation(group):
    n = len(group)
    # Random walk with small steps
    random_walk = np.cumsum(np.random.randn(n) * 0.02)
    # Keep prices bounded between 0.1 and 0.9
    group = group.copy()
    group["yes_price"] = 0.5 + np.clip(random_walk, -0.4, 0.4)
    group["no_price"] = 1 - group["yes_price"]
    return group

df_price = df_price.sort_values(["market_id", "timestamp"])
df_price = df_price.groupby("market_id", group_keys=False).apply(add_price_variation, include_groups=False)

print("Price variation check:")
print(df_price[["yes_price", "no_price"]].head(15))
print(f"\nYes price range: {df_price['yes_price'].min():.4f} - {df_price['yes_price'].max():.4f}")

Price variation check:
     yes_price  no_price
615   0.509934  0.490066
574   0.507169  0.492831
535   0.520123  0.479877
514   0.550583  0.449417
470   0.545900  0.454100
..         ...       ...
293   0.580344  0.419656
279   0.571029  0.428971
259   0.575869  0.424131
249   0.537603  0.462397
235   0.503105  0.496895

[15 rows x 2 columns]

Yes price range: 0.2968 - 0.6915


In [5]:
df_price.info()

<class 'pandas.core.frame.DataFrame'>
Index: 616 entries, 615 to 260
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   id         616 non-null    int64              
 1   yes_price  616 non-null    float64            
 2   no_price   616 non-null    float64            
 3   timestamp  616 non-null    datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(2), int64(1)
memory usage: 24.1 KB


In [6]:
df_trades.head()

,id,user_id,market_id,outcome_type,trade_type,amount_staked,tokens_amount,price_at_execution,onchain_trade_id,onchain_tx_hash,created_at
0,9518,343,43,YES,buy,5.69,9.22,0.617188,704744,0x08a6ee02814c,2025-12-01 00:44:53.699582+00:00
1,9517,343,24,NO,buy,27.48,72.52,0.378937,682598,0x0bbbaa05fa91,2025-12-01 00:14:51.699582+00:00
2,9516,343,21,NO,buy,19.25,35.01,0.549838,884372,0x02b7d408b64c,2025-11-30 23:37:36.699582+00:00
3,11367,443,41,YES,buy,18.41,33.20,0.554495,609407,0x0d269f02c62a,2025-11-30 23:35:45.772966+00:00
4,11366,443,18,NO,buy,13.51,44.56,0.303211,861836,0x03063a0d26f2,2025-11-30 23:18:43.772966+00:00


In [7]:
df_liq.head()

,id,market_id,user_id,event_type,amount,onchain_tx_hash,onchain_liquidity_id,created_at
0,192,29,354,remove,198.28,0x0e5fd00cc0f4,745035,2025-12-07 07:04:48.424405+00:00
1,240,33,65,remove,198.63,0x0f134202a44b,4868,2025-12-07 04:04:48.641717+00:00
2,239,33,294,add,108.53,0x0b31a4065d5d,782214,2025-12-07 03:04:48.641717+00:00
3,191,29,168,remove,123.38,0x03f7f60a57c4,108883,2025-12-06 17:04:48.424405+00:00
4,190,29,296,add,141.73,0x0c69e6088e7c,415641,2025-12-06 14:04:48.424405+00:00


In [ ]:
df = df_price.copy()

# 1) Mid price
df["mid_price"] = (df["yes_price"] + df["no_price"]) / 2

# 2) Sort by market and timestamp
df = df.sort_values(["market_id", "timestamp"])

# 3) Next mid_price (for target)
df["mid_price_next"] = df.groupby("market_id")["mid_price"].shift(-1)

# 4) Return for next step
df["return_next"] = (df["mid_price_next"] / df["mid_price"]) - 1.0

KeyError: 'market_id'

In [ ]:
# 5) Target label (UP/DOWN/FLAT)
up_thresh = 0.01   # +1%
down_thresh = -0.01  # -1%

def label_direction(r):
    if pd.isna(r):
        return np.nan
    if r > up_thresh:
        return "UP"
    elif r < down_thresh:
        return "DOWN"
    else:
        return "FLAT"

df["target_direction"] = df["return_next"].apply(label_direction)

df[["market_id", "timestamp", "mid_price", "mid_price_next", "return_next", "target_direction"]].head(15)

,market_id,timestamp,mid_price,mid_price_next,return_next,target_direction
615,15,2025-11-23 09:04:51.668191+00:00,0.5,0.5,0.0,FLAT
574,15,2025-11-23 13:04:51.668191+00:00,0.5,0.5,0.0,FLAT
535,15,2025-11-23 22:04:51.668191+00:00,0.5,0.5,0.0,FLAT
514,15,2025-11-24 04:04:51.668191+00:00,0.5,0.5,0.0,FLAT
470,15,2025-11-24 12:04:51.668191+00:00,0.5,0.5,0.0,FLAT
...,...,...,...,...,...,...
293,15,2025-11-26 03:04:51.668191+00:00,0.5,0.5,0.0,FLAT
279,15,2025-11-26 06:04:51.668191+00:00,0.5,0.5,0.0,FLAT
259,15,2025-11-26 11:04:51.668191+00:00,0.5,0.5,0.0,FLAT
249,15,2025-11-26 13:04:51.668191+00:00,0.5,0.5,0.0,FLAT


In [9]:
#Price-based features (log_return, rolling volatility & drift)

# Log return for current step
df["mid_price_prev"] = df.groupby("market_id")["mid_price"].shift(1)
df["log_return"] = np.log(df["mid_price"] / df["mid_price_prev"])

# Rolling window sizes (observation-based, not time-based)
WINDOW_SHORT = 12   # ~short-term volatility
WINDOW_LONG = 48    # ~longer-term volatility

# Short-term volatility (rolling std of log returns)
df["vol_short"] = (
    df.groupby("market_id")["log_return"]
      .rolling(WINDOW_SHORT, min_periods=3)
      .std()
      .reset_index(level=0, drop=True)
)

# Long-term volatility
df["vol_long"] = (
    df.groupby("market_id")["log_return"]
      .rolling(WINDOW_LONG, min_periods=10)
      .std()
      .reset_index(level=0, drop=True)
)

# Short-term drift (rolling mean of log returns)
df["drift_short"] = (
    df.groupby("market_id")["log_return"]
      .rolling(WINDOW_SHORT, min_periods=3)
      .mean()
      .reset_index(level=0, drop=True)
)



KeyError: 'market_id'

In [ ]:
# Volatility ratio (short/long) - useful for regime detection
df["vol_ratio"] = df["vol_short"] / df["vol_long"]

df[["market_id", "timestamp", "mid_price", "log_return", "vol_short", "vol_long", "drift_short", "vol_ratio"]].head(15)

,market_id,timestamp,mid_price,log_return,vol_short,vol_long,drift_short,vol_ratio
615,15,2025-11-23 09:04:51.668191+00:00,0.5,NaN,NaN,NaN,NaN,NaN
574,15,2025-11-23 13:04:51.668191+00:00,0.5,0.0,NaN,NaN,NaN,NaN
535,15,2025-11-23 22:04:51.668191+00:00,0.5,0.0,NaN,NaN,NaN,NaN
514,15,2025-11-24 04:04:51.668191+00:00,0.5,0.0,0.0,NaN,0.0,NaN
470,15,2025-11-24 12:04:51.668191+00:00,0.5,0.0,0.0,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...
293,15,2025-11-26 03:04:51.668191+00:00,0.5,0.0,0.0,0.0,0.0,NaN
279,15,2025-11-26 06:04:51.668191+00:00,0.5,0.0,0.0,0.0,0.0,NaN
259,15,2025-11-26 11:04:51.668191+00:00,0.5,0.0,0.0,0.0,0.0,NaN
249,15,2025-11-26 13:04:51.668191+00:00,0.5,0.0,0.0,0.0,0.0,NaN


In [ ]:
# Simple approach: aggregate trades into time buckets and join later

# 1) Create hourly time buckets for trades
df_trades["time_bucket"] = df_trades["created_at"].dt.floor("1h")

# 2) Aggregate trades per market per hour
trade_agg = (
    df_trades
    .groupby(["market_id", "time_bucket"], as_index=False)
    .agg(
        volume_bucket=("amount_staked", "sum"),
        trades_bucket=("amount_staked", "count"),
    )
)

trade_agg.head()

,market_id,time_bucket,volume_bucket,trades_bucket
0,15,2025-11-23 09:00:00+00:00,582.09,6
1,15,2025-11-23 10:00:00+00:00,46.47,2
2,15,2025-11-23 11:00:00+00:00,10.54,2
3,15,2025-11-23 12:00:00+00:00,80.45,2
4,15,2025-11-23 13:00:00+00:00,96.20,4


In [ ]:
# Create hourly time bucket for PriceHistory
df["time_bucket"] = df["timestamp"].dt.floor("1h")

# Merge trade aggregates with price data
df = df.merge(
    trade_agg,
    how="left",
    on=["market_id", "time_bucket"],
)

# Fill NaN volume/trades with 0 (no trades in that bucket)
df["volume_bucket"] = df["volume_bucket"].fillna(0)
df["trades_bucket"] = df["trades_bucket"].fillna(0)

df[["market_id", "timestamp", "volume_bucket", "trades_bucket"]].head(15)

,market_id,timestamp,volume_bucket,trades_bucket
0,15,2025-11-23 09:04:51.668191+00:00,582.09,6.0
1,15,2025-11-23 13:04:51.668191+00:00,96.20,4.0
2,15,2025-11-23 22:04:51.668191+00:00,0.00,0.0
3,15,2025-11-24 04:04:51.668191+00:00,0.00,0.0
4,15,2025-11-24 12:04:51.668191+00:00,98.88,3.0
...,...,...,...,...
10,15,2025-11-26 03:04:51.668191+00:00,0.00,0.0
11,15,2025-11-26 06:04:51.668191+00:00,0.00,0.0
12,15,2025-11-26 11:04:51.668191+00:00,93.38,3.0
13,15,2025-11-26 13:04:51.668191+00:00,67.26,3.0


In [ ]:
df_liq["time_bucket"] = df_liq["created_at"].dt.floor("1h")

event_sign = {"ADD": 1, "REMOVE": -1}
df_liq["sign"] = df_liq["event_type"].map(event_sign).fillna(0)
df_liq["liq_effect"] = df_liq["sign"] * df_liq["amount"]

liq_agg = (
    df_liq
    .groupby(["market_id", "time_bucket"], as_index=False)
    .agg(
        liquidity_delta_bucket=("liq_effect", "sum"),
        liquidity_events_bucket=("amount", "count"),
    )
)

df = df.merge(
    liq_agg,
    how="left",
    on=["market_id", "time_bucket"],
)

for c in ["liquidity_delta_bucket", "liquidity_events_bucket"]:
    df[c] = df[c].fillna(0)

df[["market_id", "timestamp", "liquidity_delta_bucket", "liquidity_events_bucket"]].head(15)


,market_id,timestamp,liquidity_delta_bucket,liquidity_events_bucket
0,15,2025-11-23 09:04:51.668191+00:00,0.0,1.0
1,15,2025-11-23 13:04:51.668191+00:00,0.0,0.0
2,15,2025-11-23 22:04:51.668191+00:00,0.0,0.0
3,15,2025-11-24 04:04:51.668191+00:00,0.0,0.0
4,15,2025-11-24 12:04:51.668191+00:00,0.0,0.0
...,...,...,...,...
10,15,2025-11-26 03:04:51.668191+00:00,0.0,0.0
11,15,2025-11-26 06:04:51.668191+00:00,0.0,0.0
12,15,2025-11-26 11:04:51.668191+00:00,0.0,0.0
13,15,2025-11-26 13:04:51.668191+00:00,0.0,1.0


In [ ]:
# Select feature columns
feature_cols = [
    "log_return",
    "vol_short",
    "vol_long",
    "drift_short",
    "volume_bucket",
    "trades_bucket",
    "liquidity_delta_bucket",
    "liquidity_events_bucket",
]

target_col = "target_direction"

# Keep only rows where target and all features are not NaN
df_model = df.dropna(subset=feature_cols + [target_col]).copy()

# Check class distribution
print(df_model[target_col].value_counts())
df_model[feature_cols].head()

target_direction
FLAT    287
Name: count, dtype: int64


,log_return,vol_short,vol_long,drift_short,volume_bucket,trades_bucket,liquidity_delta_bucket,liquidity_events_bucket
10,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
11,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,93.38,3.0,0.0,0.0
13,0.0,0.0,0.0,0.0,67.26,3.0,0.0,1.0
14,0.0,0.0,0.0,0.0,3.32,1.0,0.0,0.0


In [10]:
# Sort by market and timestamp
df_model = df_model.sort_values(["market_id", "timestamp"])

# For simplicity, treat all markets as a single time series
# (can train separately per market later if needed)

X = df_model[feature_cols].values
y = df_model[target_col].values

# Time-based split: last 20% for testing
split_idx = int(len(df_model) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

NameError: name 'df_model' is not defined

In [19]:
# Check the actual return distribution
print("Return statistics:")
print(df["return_next"].describe())
print("\nReturn quantiles:")
print(df["return_next"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

Return statistics:
count    586.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: return_next, dtype: float64

Return quantiles:
0.10    0.0
0.25    0.0
0.50    0.0
0.75    0.0
0.90    0.0
Name: return_next, dtype: float64


In [15]:
# Check if all classes exist in training data
print("Training set class counts:")
print(pd.Series(y_train).value_counts())

Training set class counts:
FLAT    229
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(df_model[target_col].values)

# Check class distribution
print("Classes:", le.classes_)
print("Train class distribution:", np.bincount(y_encoded[:split_idx]))
print("Test class distribution:", np.bincount(y_encoded[split_idx:]))

# Split with encoded labels
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y_encoded[:split_idx], y_encoded[split_idx:]

Classes: ['FLAT']
Train class distribution: [229]
Test class distribution: [58]


In [18]:
# ===============================
# Stacking Ensemble: RF + XGBoost + LogisticRegression
# ===============================

# Base learners
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1,
)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",  # multi-class
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

logreg_base = LogisticRegression(
    max_iter=1000,
    multi_class="auto",
    n_jobs=-1,
)

base_estimators = [
    ("rf", rf),
    ("xgb", xgb),
    ("lr", logreg_base),
]

# Final meta-learner (top layer)
meta_learner = LogisticRegression(
    max_iter=1000,
    multi_class="auto",
    n_jobs=-1,
)

from sklearn.model_selection import StratifiedKFold

stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),  # Stratified CV
    n_jobs=-1,
    passthrough=False,
)

# Train
stack_clf.fit(X_train, y_train)

# Predictions
y_pred = stack_clf.predict(X_test)
y_proba = stack_clf.predict_proba(X_test)

print("=== Classification Report (Stacking Ensemble) ===")
print(classification_report(y_test, y_pred))

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)